# Milestone 2 — Full parser/BibTeX pipeline trên Google Colab

**Nhánh:** `m3-data-split`. Notebook này thực hiện **Milestone 2**, không phải thuật toán matching/split của Milestone 3.

1. Chọn **Runtime → Change runtime type → CPU**.
2. Sửa `CORPUS_DIRS` và `RUN_DIR` ở cell cấu hình. Corpus là dữ liệu M1 **đã có**, không crawl.
3. Chọn **Run all**. Notebook clone đúng branch, cài dependencies M2, parse full input, audit và xuất đủ ba dataset `.parquet`.
4. Output JSON được lưu theo paper trong `parsed/`. Khi citation key có reference mâu thuẫn, vẫn giữ cây và tất cả ứng viên, để citation mơ hồ chưa được resolve và báo **partial**.

Dữ liệu không nằm trong GitHub. Phải đặt corpus trên Drive hoặc upload/giải nén trước. Chỉ thiếu input/config hoặc lỗi hạ tầng mới dừng luồng; không tạo dữ liệu giả cho range chưa được cung cấp.


In [ ]:
# Chỉ cần sửa cấu hình tại đây trước khi Run all.
CONFIG = {
    "REPO_URL": "https://github.com/VincesPowder/hcmus-project-arxiv.git",
    "BRANCH": "m3-data-split",
    "REPO_DIR": "/content/hcmus-project-arxiv",
    "CORPUS_DIRS": [
        "/content/drive/MyDrive/arXiv/corpus",
        # Thêm các corpus khác nếu đủ bốn range:
        # "/content/drive/MyDrive/arXiv/range_2605/corpus",
        # "/content/drive/MyDrive/arXiv/range_2602/corpus",
        # "/content/drive/MyDrive/arXiv/range_2512/corpus",
    ],
    "RUN_DIR": "/content/drive/MyDrive/IntroDS/milestone2_outputs/m2_full_v1",
    "USE_DRIVE": True,
    "WORKERS": 2,
    "TIMEOUT": 120,
    "BIB_SCOPE": "cited",       # "all" giữ mọi entry của thư viện đã khai báo
    "LIMIT": 0,                # 0 = full input; đặt 5 để thử trước
    "SHARD_SIZE": 250,
    "EXPECTED_PUBLICATIONS": 20000,
    "RETRY_INCOMPLETE": True,
    "SYNC_REPOSITORY": True,
    "INSTALL_DEPENDENCIES": True,
    "MAKE_PLOTS": True,
    "DOWNLOAD_REPORTS": False,
}
# Dùng cho cấu hình nâng cao/kiểm thử; mặc định không cần sửa.
CONFIG.update(globals().get("CONFIG_OVERRIDES", {}))

import json
import os
import subprocess
import sys
from pathlib import Path

REPO = Path(CONFIG["REPO_DIR"]).expanduser().resolve()
RUN = Path(CONFIG["RUN_DIR"]).expanduser().resolve()
print("Branch:", CONFIG["BRANCH"])
print("Input directories:", CONFIG["CORPUS_DIRS"])
print("Output:", RUN)
print("Mode:", "FULL INPUT" if CONFIG["LIMIT"] == 0 else f'LIMIT {CONFIG["LIMIT"]}')


## 1. Clone đúng branch và cài môi trường

Cell này không dùng clone URL placeholder hoặc clone nhánh mặc định. Repo runtime có thay đổi local/diverged sẽ bị từ chối cập nhật tự động; không reset hoặc xóa file của bạn.


In [ ]:
def command(args, **kwargs):
    try:
        return subprocess.run(args, check=True, **kwargs)
    except subprocess.CalledProcessError as error:
        if error.stderr:
            print(error.stderr)
        raise

if CONFIG["SYNC_REPOSITORY"]:
    if not REPO.exists():
        command(["git", "clone", "--single-branch", "--branch", CONFIG["BRANCH"],
                 CONFIG["REPO_URL"], str(REPO)])
    else:
        if not (REPO / ".git").exists():
            raise RuntimeError(f"{REPO} exists but is not a Git checkout. Choose a new REPO_DIR.")
        dirty = subprocess.check_output(
            ["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
        if dirty:
            raise RuntimeError("Runtime checkout has tracked local changes. Preserve them and choose a new REPO_DIR.")
        origin = subprocess.check_output(
            ["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
        normalize_url = lambda value: value.rstrip("/").removesuffix(".git")
        if normalize_url(origin) != normalize_url(CONFIG["REPO_URL"]):
            raise RuntimeError("REPO_DIR points to a different repository. Choose a new directory.")
        command(["git", "-C", str(REPO), "fetch", "origin", CONFIG["BRANCH"]])
        current = subprocess.check_output(
            ["git", "-C", str(REPO), "branch", "--show-current"], text=True).strip()
        if current != CONFIG["BRANCH"]:
            exists = subprocess.run(["git", "-C", str(REPO), "show-ref", "--verify", "--quiet",
                                     "refs/heads/" + CONFIG["BRANCH"]]).returncode == 0
            if exists:
                command(["git", "-C", str(REPO), "switch", CONFIG["BRANCH"]])
            else:
                command(["git", "-C", str(REPO), "switch", "-c", CONFIG["BRANCH"], "FETCH_HEAD"])
        command(["git", "-C", str(REPO), "merge", "--ff-only", "FETCH_HEAD"])

if not (REPO / "src/tools/run_m2_full.py").is_file():
    raise RuntimeError("Missing full runner. Check REPO_DIR and fetch the updated m3-data-split branch.")
os.chdir(REPO)
if CONFIG["INSTALL_DEPENDENCIES"]:
    command([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "src/requirements-m2.txt")])

ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "src") + os.pathsep + ENV.get("PYTHONPATH", "")
print("Python:", sys.version)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 2. Mount Drive và kiểm tra layout đầu vào

Mỗi phần tử của `CORPUS_DIRS` phải là thư mục chứa các publication, ví dụ `.../corpus` hoặc `.../work/raw`.

Hai layout được hỗ trợ:

```text
corpus/<paper>/paper.json + vN/metadata.json + *.tex/*.bib
work/raw/<paper>/paper_metadata.json + sources/vN/manifest.json + *.tex/*.bib
```

Không trỏ vào file ZIP, thư mục một version, thư mục `parsed` hoặc thư mục cha còn chứa nhiều archive. Không chạy đồng thời hai runtime ghi vào cùng `RUN_DIR`.


In [ ]:
if CONFIG["USE_DRIVE"]:
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError("USE_DRIVE=True requires Colab. Set False for local execution.") from error
    drive.mount("/content/drive")

import shutil
inspection = (
    "import json,sys; from tools.run_m2_full import inventory; "
    "print(json.dumps(inventory(sys.argv[1:]),ensure_ascii=False))"
)
inventory_result = command(
    [sys.executable, "-c", inspection, *CONFIG["CORPUS_DIRS"]],
    cwd=REPO, env=ENV, capture_output=True, text=True,
)
INPUT = json.loads(inventory_result.stdout)
print("Input publications:", INPUT["publications"])
print("Declared versions:", INPUT["declared_versions"])
print("Examples:", [(p["arxiv_id"], p["versions"]) for p in INPUT["papers"][:3]])
if CONFIG["EXPECTED_PUBLICATIONS"] and INPUT["publications"] != CONFIG["EXPECTED_PUBLICATIONS"]:
    print("NOTICE: input differs from the 20,000-publication group target. Coverage will show missing output.")
existing_parent = RUN
while not existing_parent.exists():
    existing_parent = existing_parent.parent
print("Available output filesystem space (GiB):", round(shutil.disk_usage(existing_parent).free / 1024**3, 2))


## 3. Parse full → audit → xuất Parquet

Flow thực thi:

```text
metadata từng publication → từng version → gather compilation path
→ hierarchy + cleanup/math normalization
→ bibitem/.bib extraction → reference field union + aliases theo version
→ đổi cite → full-text dedup → parsed/<paper>.json
→ audit → exports/<lượt xuất mới>/tables/{elements,hierarchy,bibtex}/*.parquet
```

`LIMIT=0` xử lý toàn bộ input đã chọn. Runner dùng CLI subprocess để multiprocessing và timeout không phụ thuộc trạng thái kernel notebook.

JSON được ghi vào `RUN_DIR/parsed` để resume. Thay parser, input selection, timeout hoặc bib scope cần `RUN_DIR` mới. Input không được sửa. Các version empty/not_downloaded/no_root/timeout/parse_error và publication reference-conflict được báo riêng.

Nếu còn paper/source lỗi, trạng thái là `partial` và các cell thống kê/xuất dữ liệu vẫn chạy. Một code return 0 ở bước orchestration không có nghĩa toàn bộ 20.000 bài đã hoàn tất.


In [ ]:
args = [
    sys.executable, "-u", str(REPO / "src/tools/run_m2_full.py"),
    "--corpus-dirs", *CONFIG["CORPUS_DIRS"],
    "--run-dir", str(RUN),
    "--workers", str(CONFIG["WORKERS"]),
    "--timeout", str(CONFIG["TIMEOUT"]),
    "--bib-scope", CONFIG["BIB_SCOPE"],
    "--limit", str(CONFIG["LIMIT"]),
    "--shard-size", str(CONFIG["SHARD_SIZE"]),
    "--expected-publications", str(CONFIG["EXPECTED_PUBLICATIONS"]),
]
if not CONFIG["RETRY_INCOMPLETE"]:
    args.append("--no-retry-incomplete")
command(args, cwd=REPO, env=ENV)
RESULT = json.loads((RUN / "latest_result.json").read_text(encoding="utf-8"))
SUMMARY = RESULT["summary"]
METRICS = RESULT["metrics"]
print("\nRUN STATUS:", RESULT["status"])
print("Selected publications:", SUMMARY["selected_publications"])
print("Paper outputs:", SUMMARY["publications"])
print("Quarantined publications:", SUMMARY.get("quarantined_publications", 0))
print("Versions without paper output:", SUMMARY["versions_without_paper_output"])


## 4. Coverage và counts trước/sau dedup

`processed` theo range là số publication có JSON, không phải số publication có mọi version parsed. Root riêng từng version có trong bảng elements; counts dedup bên dưới không tính roots.

`bib_before_dedup` là entry đầu vào `unify_bibtex` sau bước lọc cite và đối chiếu bibitem/.bib cùng key, không phải tất cả entry thô trong các thư viện.


In [ ]:
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

display(pd.DataFrame(SUMMARY["range_coverage"]))
display(pd.DataFrame(
    sorted(SUMMARY["version_statuses"].items()), columns=["version_status", "count"]))
totals = SUMMARY["totals"]
display(pd.DataFrame([
    {"type": "elements (excluding roots)", "before": totals.get("nodes_before_dedup", 0),
     "after": totals.get("elements_after_dedup", 0)},
    {"type": "references", "before": totals.get("bib_before_dedup", 0),
     "after": totals.get("bib_after_dedup", 0)},
]))
versions = pd.read_csv(RUN / "audit/coverage.csv", dtype={"arxiv_id": str})
publications = pd.read_csv(RUN / "audit/publications.csv", dtype={"arxiv_id": str})
print("Version coverage example:")
display(versions.head(10))
print("Publication statistics example:")
display(publications.head(10))
observed = sum(SUMMARY["version_statuses"].values())
declared = SUMMARY["selected_declared_versions"]
parsed_count = SUMMARY["version_statuses"].get("parsed", 0)
print("Observed version records:", observed, "/ declared selected:", declared)
print("Parsed / declared selected:", f"{parsed_count / declared:.2%}" if declared else "N/A")
print("Hierarchy problems:", totals.get("hierarchy_problems", 0))


## 5. Kiểm tra đủ ba bảng Parquet và xem mẫu

Có reference hoặc element bằng 0 vẫn có dataset rỗng đúng schema. Nếu mọi publication đều failed, không tạo output giả; notebook hiển thị lỗi và bỏ qua preview dữ liệu không tồn tại.


In [ ]:
import pyarrow.dataset as ds

EXPORT = RESULT.get("export")
if EXPORT:
    TABLES = Path(EXPORT["directory"]) / "tables"
    CHECK = json.loads((Path(EXPORT["directory"]) / "export_validation.json").read_text(encoding="utf-8"))
    print("Verified export:", CHECK)
    for name in ("elements", "hierarchy", "bibtex"):
        dataset = ds.dataset(TABLES / name, format="parquet")
        print("\nTABLE:", name, "| rows:", dataset.count_rows())
        print(dataset.schema)
        display(dataset.head(5).to_pandas())
else:
    TABLES = None
    print("No successful paper output: inspect paper_failures.json; no fabricated Parquet rows.")


## 6. Reference quality và các lỗi cần review

Missing DOI/title/authors có thể do input thiếu hoặc heuristic chưa nhận diện; không tự suy ra accuracy từ tỷ lệ field đầy đủ.

`unresolved_citation_occurrences` đếm bộ `(publication, version, key)`, không đếm mỗi lần cite lặp trong cùng version. `field_conflicts`, `extraction_conflicts` và source keys được giữ trong JSON/JSONL; Parquet giữ schema của đề.

Key mâu thuẫn trong cùng version có danh sách `candidate_keys` tại `audit/reference_conflicts.jsonl` và `reference_conflicts` trong JSON từng bài. Mỗi ứng viên có key riêng cùng `ambiguous_source_keys`; citation gốc được giữ chưa resolve, không chọn tùy tiện một ứng viên. Dù không còn publication failed, trạng thái vẫn là `partial` nếu còn xung đột này.


In [ ]:
print("Quality flags:")
display(pd.DataFrame(sorted(SUMMARY["quality_flags"].items()), columns=["flag", "count"]))
print("Unresolved (publication, version, key) pairs:", SUMMARY["unresolved_citation_occurrences"])
failures = json.loads((RUN / "paper_failures.json").read_text(encoding="utf-8"))
print("Paper failures:", len(failures))
display(pd.DataFrame(failures).head(10))
unresolved = pd.read_csv(RUN / "audit/unresolved_citations.csv", dtype={"arxiv_id": str})
display(unresolved.head(10))
print("Ambiguous citation keys (candidate references retained):")
with (RUN / "audit/reference_conflicts.jsonl").open(encoding="utf-8") as handle:
    for index, line in enumerate(handle):
        print(json.loads(line))
        if index >= 9:
            break
print("Reference review examples:")
with (RUN / "audit/reference_review.jsonl").open(encoding="utf-8") as handle:
    for index, line in enumerate(handle):
        print(json.loads(line))
        if index >= 2:
            break


## 7. Thời gian, RAM và biểu đồ từ output thực

`wall_seconds` của runner gồm parse + audit + export; không gồm clone/cài packages, vẽ hình và ZIP. `avg_seconds_per_paper` là parse time chia số paper không skipped trong lượt này; lượt resume không được báo như benchmark fresh.

Kiểm tra `ram_scope=process_tree` trước khi gọi RAM là tổng workers. RSS lấy mẫu có thể bỏ lỡ peak và đếm lặp bộ nhớ chia sẻ. Chỉ công bố benchmark Colab CPU khi chính lượt chạy thực sự dùng môi trường đó.


In [ ]:
print(json.dumps(METRICS, ensure_ascii=False, indent=2))
if CONFIG["MAKE_PLOTS"]:
    import matplotlib.pyplot as plt
    FIGURES = RUN / "figures"
    FIGURES.mkdir(exist_ok=True)
    states = SUMMARY["version_statuses"]
    if states:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(list(states), list(states.values()))
        ax.set(title="M2 version coverage", ylabel="Versions")
        ax.tick_params(axis="x", rotation=25)
        fig.tight_layout()
        fig.savefig(FIGURES / "version_status.png", dpi=160)
        display(fig)
        plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, label, before, after in [
        (axes[0], "Elements (no roots)", totals.get("nodes_before_dedup", 0), totals.get("elements_after_dedup", 0)),
        (axes[1], "References", totals.get("bib_before_dedup", 0), totals.get("bib_after_dedup", 0)),
    ]:
        ax.bar(["Before", "After"], [before, after])
        ax.set(title=label, ylabel="Rows")
    fig.tight_layout()
    fig.savefig(FIGURES / "dedup_counts.png", dpi=160)
    display(fig)
    plt.close(fig)


## 8. Files đã lưu và tải báo cáo

`reports.zip` là gói báo cáo nhỏ (counts, coverage, failures, metrics, manifest và schema checks), **không chứa toàn bộ JSON/Parquet**. Dữ liệu đầy đủ vẫn nằm trong `RUN_DIR` trên Drive.

Giữ `parsed/` vì JSON chứa provenance/conflicts và có thể dựng lại bảng không cần parse source. Giữ các lượt export ở thư mục riêng; không trộn shard cũ/mới hoặc chép đè shard cùng tên của nhiều range.


In [ ]:
REPORT_ZIP = RUN / "reports.zip"
print("RUN STATUS:", RESULT["status"])
print("Parsed JSON:", RUN / "parsed")
print("Audit:", RUN / "audit")
print("Metrics:", RUN / "work/parse_metrics.json")
print("Failures:", RUN / "paper_failures.json")
print("Latest result:", RUN / "latest_result.json")
print("Parquet:", TABLES if TABLES else "not generated (all publications failed)")
print("Reports ZIP:", REPORT_ZIP, "| exists:", REPORT_ZIP.is_file())
print("JSON files:", len(list((RUN / "parsed").glob("*.json"))))
if TABLES:
    for name in ("elements", "hierarchy", "bibtex"):
        files = sorted((TABLES / name).glob("*.parquet"))
        print(name, "shards:", len(files), "| first:", files[0].name if files else "none")
if CONFIG["DOWNLOAD_REPORTS"]:
    from google.colab import files
    files.download(str(REPORT_ZIP))


## Checklist hoàn tất

- Có output thực của đủ các range cần báo cáo; không ngoại suy range thiếu.
- Đối chiếu `selected_declared_versions` với version records và `versions_without_paper_output`.
- Review `paper_failures.json`, unresolved citations và metadata conflicts.
- Xác minh ba bảng đã qua schema/count checks.
- Ghi cấu hình, commit, fingerprint và trạng thái fresh/resume khi báo cáo hiệu năng.
- Chỉ coi nhóm hoàn tất khi đã xử lý phần thiếu/lỗi, không chỉ vì notebook chạy hết các cell.

Không có cell crawler trong notebook này. Thư mục `parsed` và `exports` là dữ liệu dùng tiếp cho M3; không commit raw corpus hoặc output dung lượng lớn vào Git.
